In [ ]:
from ortools.sat.python import cp_model

N = 5

start = (1, 1)
target = (4, 4)

obstacles = [(2, 2)]

MAX_STEPS = 6

model = cp_model.CpModel()

x = []
y = []

for i in range(MAX_STEPS):
    x.append(model.NewIntVar(0, N - 1, f'x{i}'))
    y.append(model.NewIntVar(0, N - 1, f'y{i}'))

model.Add(x[0] == start[0])
model.Add(y[0] == start[1])

model.Add(x[MAX_STEPS - 1] == target[0])
model.Add(y[MAX_STEPS - 1] == target[1])

for i in range(MAX_STEPS - 1):
    dx = model.NewIntVar(-1, 1, f'dx{i}')
    dy = model.NewIntVar(-1, 1, f'dy{i}')

    model.Add(dx == x[i+1] - x[i])
    model.Add(dy == y[i+1] - y[i])

    model.AddAbsEquality(1, dx)
    model.AddAbsEquality(1, dy)

for i in range(MAX_STEPS):
    for ox, oy in obstacles:
        model.AddForbiddenAssignments([x[i], y[i]], [(ox, oy)])

solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.FEASIBLE or status == cp_model.OPTIMAL:
    print("path:")
    for i in range(MAX_STEPS):
        print(f"({solver.Value(x[i])},{solver.Value(y[i])})")
else:
    print("no path found")

path:
(1,1)
(0,2)
(1,3)
(2,4)
(3,3)
(4,4)


In [3]:
from ortools.sat.python import cp_model

grid = [
    [1,1,0,0,0],
    [1,1,0,1,1],
    [0,0,0,1,1],
    [0,1,1,0,0],
    [0,1,1,0,0]
]

rows = len(grid)
cols = len(grid[0])

model = cp_model.CpModel()

cell = {}
for i in range(rows):
    for j in range(cols):
        cell[i,j] = model.NewBoolVar(f'cell_{i}_{j}')
        model.Add(cell[i,j] == grid[i][j])

perimeter_edges = []

directions = [(1,0),(-1,0),(0,1),(0,-1)]

for i in range(rows):
    for j in range(cols):
        if grid[i][j] == 1:

            for dx,dy in directions:
                ni = i + dx
                nj = j + dy

                edge = model.NewBoolVar(f'edge_{i}_{j}_{dx}_{dy}')

                # boundary case
                if ni < 0 or nj < 0 or ni >= rows or nj >= cols:
                    model.Add(edge == 1)

                else:
                    if grid[ni][nj] == 0:
                        model.Add(edge == 1)
                    else:
                        model.Add(edge == 0)

                perimeter_edges.append(edge)

perimeter = model.NewIntVar(0,1000,'perimeter')
model.Add(perimeter == sum(perimeter_edges))

solver = cp_model.CpSolver()
solver.Solve(model)

print("Perimeter =", solver.Value(perimeter))

Perimeter = 24


In [ ]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

distance_matrix = [
    [0,29,20,21,16,31,100,12,4,31],
    [29,0,15,29,28,40,72,21,29,41],
    [20,15,0,15,14,25,81,9,23,27],
    [21,29,15,0,4,12,92,12,25,13],
    [16,28,14,4,0,16,94,9,20,16],
    [31,40,25,12,16,0,95,24,36,3],
    [100,72,81,92,94,95,0,90,101,99],
    [12,21,9,12,9,24,90,0,15,25],
    [4,29,23,25,20,36,101,15,0,35],
    [31,41,27,13,16,3,99,25,35,0]
]

def create_data_model():
    data = {}
    data['distance_matrix'] = distance_matrix
    data['num_vehicles'] = 1
    data['depot'] = 0
    return data

data = create_data_model()

manager = pywrapcp.RoutingIndexManager(
    len(data['distance_matrix']),
    data['num_vehicles'],
    data['depot'])

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

solution = routing.SolveWithParameters(search_parameters)

if solution:
    index = routing.Start(0)
    route = []
    route_distance = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route.append(node)
        previous_index = index
        index = solution.Value(routing.NextVar(index))
        route_distance += routing.GetArcCostForVehicle(
            previous_index, index, 0)

    route.append(manager.IndexToNode(index))

    print("optimal path:")
    print(route)
    print("total distance:", route_distance)

optimal path:
[0, 8, 7, 2, 1, 6, 5, 9, 3, 4, 0]
total distance: 246
